In [ ]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Definir la ruta de tu carpeta en Drive
ruta_drive = '/content/drive/MyDrive/Archivos_Jammes/Proyecto/BiomasAndinos/'

# Verificar que los archivos estén ahí
if os.path.exists(ruta_drive):
    print(f"✅ Carpeta encontrada. Archivos detectados: {os.listdir(ruta_drive)}")
else:
    print("❌ No se encontró la carpeta. Revisa el nombre en tu Google Drive.")

In [ ]:


ee.Authenticate()

ee.Initialize(project='project-5f1624a1-ea37-4b43-b7e')

In [ ]:
import ee
import geemap
import pandas as pd
import os
from datetime import datetime, timedelta

# 1. Configuration and Setup
bandas_ml = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'NDVI', 'NDMI', 'BSI']
roi = geemap.shp_to_ee(ruta_drive + 'acha_poligono.shp')
ruta_raw_data = os.path.join(ruta_drive, 'RawData')

if not os.path.exists(ruta_raw_data):
    os.makedirs(ruta_raw_data)

# Removed config_biomas, colecciones_puntos, entrenamiento_puntos as requested.

# 2. Generate a list of dates for the specified range (2015-01-01 to 2026-03-31) with a 5-day step
def generate_dates_range(start_year, end_year, end_month, end_day, step_days=5):
    dates = []
    for year in range(start_year, end_year + 1):
        start = datetime(year, 1, 1)
        # Adjust end date for the final year, otherwise use end of year
        if year == end_year:
            end = datetime(year, end_month, end_day)
        else:
            end = datetime(year, 12, 31)

        current = start
        while current <= end:
            dates.append(current.strftime('%Y-%m-%d'))
            current += timedelta(days=step_days)
    return dates

fechas_interes = generate_dates_range(2015, 2026, 3, 31, step_days=5)

print(f"Starting full 2015-01-01 to 2026-03-31 extraction with shortest window (5 days). Total passes to check: {len(fechas_interes)}")

for fecha in fechas_interes:
    start_date = ee.Date(fecha)
    end_date = start_date.advance(5, 'day')

    # Fetch ALL available images in the 5-day window and create a median composite
    img_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                      .filterBounds(roi)
                      .filterDate(start_date, end_date))

    # Check if there are any images in the collection before taking the median
    if img_collection.size().getInfo() == 0:
        continue

    img = img_collection.median()

    # Get the actual sensing date from the image metadata and cloud percentage (from the composite)
    # For a median composite, CLOUDY_PIXEL_PERCENTAGE might not be directly available or meaningful for the composite itself.
    # We'll use the original cloud filter value (40) as a reference, or you can calculate average cloudiness if needed.
    actual_date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo() if img.get('system:time_start').getInfo() else fecha # Fallback to window start date
    # For a median composite, individual image cloud percentages are averaged or ignored. Use a placeholder or calculate if necessary.
    cloud_percentage = 0 # Placeholder, as exact cloud percentage for a composite is complex without further calculation

    output_name = f'dataset_satellite_raw_{actual_date}.csv' # Changed output name
    full_path = os.path.join(ruta_raw_data, output_name)

    # Skip if file already exists to save time
    if os.path.exists(full_path):
        print(f"- {actual_date} already exists. Skipping.")
        continue

    img = img.clip(roi)
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndmi = img.normalizedDifference(['B8', 'B11']).rename('NDMI')
    bsi = img.expression('((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))', {
        'SWIR': img.select('B11'), 'RED': img.select('B4'),
        'NIR': img.select('B8'), 'BLUE': img.select('B2')
    }).rename('BSI')

    # No LAI calculation as requested
    img_final = img.addBands([ndvi, ndmi, bsi])

    # Sample regions directly from the ROI, without classes
    muestras = img_final.select(bandas_ml).sample(
        region=roi,
        scale=10,
        geometries=True,
        numPixels=1000 # Sample a fixed number of pixels to get representative values
    ).map(lambda f: f.set({
        'CLOUD_PERCENTAGE': cloud_percentage # Add cloud percentage to each feature
    }))


    df_date = geemap.ee_to_df(muestras)

    if not df_date.empty:
        df_date.to_csv(full_path, index=False)
        print(f"✅ Saved pass for: {actual_date}")

print("\nFull 2015-01-01 to 2026-03-31 extraction finished.")